# 04 - Silver Transformation
Cleans, validates, standardizes, and deduplicates Bronze data.
Records failing hard rules go to `silver_quarantine` instead of being dropped.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

BRONZE_TABLE = "sentinel_catalog.sentinel_schema.bronze_transactions"
SILVER_TABLE = "sentinel_catalog.sentinel_schema.silver_transactions"
QUARANTINE_TABLE = "sentinel_catalog.sentinel_schema.silver_quarantine"

bronze_df = spark.table(BRONZE_TABLE)

### Step 1: standardize / trim / cast (BEFORE validation)

In [0]:
cleaned_df = (
    bronze_df
    .withColumn("transaction_id", F.trim(F.col("transaction_id")))
    .withColumn("sender_upi_id", F.lower(F.trim(F.col("sender_upi_id"))))
    .withColumn("receiver_upi_id", F.lower(F.trim(F.col("receiver_upi_id"))))
    .withColumn(
        "amount_clean",
        F.regexp_extract(F.col("amount").cast("string"), r"-?\d+\.?\d*", 0).cast(DoubleType())
    )
    .withColumn("status_clean", F.upper(F.trim(F.col("status"))))
    .withColumn(
        "status_clean",
        F.when(F.col("status_clean").isin("SUCCESS", "FAILED", "TIMEOUT", "PENDING"), F.col("status_clean"))
         .otherwise(F.lit("UNKNOWN"))
    )
    .withColumn("event_timestamp", F.to_timestamp("timestamp"))
    .withColumn("ip_address_clean", F.coalesce(F.col("ip_address"), F.lit("UNKNOWN")))
    .withColumn(
        "customer_phone_masked",
        F.concat(F.lit("*" * 7), F.substring(F.col("customer_phone"), -2, 2))
    )
)

### Step 2: split valid vs quarantine based on hard rules

In [0]:
is_valid = (
    F.col("transaction_id").isNotNull() &
    F.col("amount_clean").isNotNull() &
    (F.col("amount_clean") > 0) &
    F.col("event_timestamp").isNotNull()
)

quarantine_df = (
    cleaned_df.filter(~is_valid)
    .withColumn(
        "rejection_reason",
        F.when(F.col("transaction_id").isNull(), F.lit("NULL_TRANSACTION_ID"))
         .when(F.col("amount_clean").isNull(), F.lit("UNPARSEABLE_AMOUNT"))
         .when(F.col("amount_clean") <= 0, F.lit("NEGATIVE_OR_ZERO_AMOUNT"))
         .when(F.col("event_timestamp").isNull(), F.lit("UNPARSEABLE_TIMESTAMP"))
         .otherwise(F.lit("OTHER"))
    )
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_df = cleaned_df.filter(is_valid).dropDuplicates(["transaction_id"])

silver_df = valid_df.select(
    F.col("transaction_id"),
    F.col("event_timestamp"),
    F.col("sender_upi_id"),
    F.col("receiver_upi_id"),
    F.col("customer_phone_masked").alias("customer_phone"),
    F.col("bank_name"),
    F.col("amount_clean").alias("amount"),
    F.col("status_clean").alias("status"),
    F.col("ip_address_clean").alias("ip_address"),
    F.col("ingestion_timestamp"),
)

In [0]:
(silver_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SILVER_TABLE))

(quarantine_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(QUARANTINE_TABLE))

print(f"Silver valid rows: {silver_df.count()}")
print(f"Quarantined rows:  {quarantine_df.count()}")

Silver valid rows: 14580
Quarantined rows:  490


### Before / after example

In [0]:
display(bronze_df.select("transaction_id", "sender_upi_id", "amount", "status").limit(5))
display(silver_df.select("transaction_id", "sender_upi_id", "amount", "status").limit(5))

transaction_id,sender_upi_id,amount,status
4e8a195c-291f-40b9-85ef-5d8da93c3ca3,lemvaa35@okicici,19014.44,SUCCESS
89123cda-01b4-4f74-85a9-b419bdcacd5a,fmngmv874@okhdfc,37682.88,TIMEOUT
4efdfb1a-0f77-4145-bd80-09c800df30f0,opkvzj838@okicici,25145.98,TIMEOUT
27d931f6-1182-4abe-99b2-69270ec4f4a0,viyqgf546@oksbi,23784.64,SUCCESS
f3320d40-f0d4-43ed-a72f-64d8fe33b483,auimuh42@okicici,36364.88 INR,PENDING


transaction_id,sender_upi_id,amount,status
4e8a195c-291f-40b9-85ef-5d8da93c3ca3,lemvaa35@okicici,19014.44,SUCCESS
89123cda-01b4-4f74-85a9-b419bdcacd5a,fmngmv874@okhdfc,37682.88,TIMEOUT
4efdfb1a-0f77-4145-bd80-09c800df30f0,opkvzj838@okicici,25145.98,TIMEOUT
27d931f6-1182-4abe-99b2-69270ec4f4a0,viyqgf546@oksbi,23784.64,SUCCESS
f3320d40-f0d4-43ed-a72f-64d8fe33b483,auimuh42@okicici,36364.88,PENDING


### Verify

In [0]:
display(spark.sql(f"SELECT rejection_reason, COUNT(*) AS cnt FROM {QUARANTINE_TABLE} GROUP BY rejection_reason ORDER BY cnt DESC"))

rejection_reason,cnt
NEGATIVE_OR_ZERO_AMOUNT,252
NULL_TRANSACTION_ID,238
